# Composing Hamiltonians: Jaynes-Cummings from parts

The term layer (`htdse.term`) makes Hamiltonians composable **by subsystem name**: a spin term
stays 2-dimensional in its definition, a motional term stays $(n_{max}+1)$-dimensional, and `+`
takes care of the registry union and identity-padding. The joint matrix only exists when an
evolution asks for `H(t)`.

$$ H = \underbrace{\tfrac{\omega_0}{2}\sigma_z}_{\text{atom}}
     + \underbrace{\omega\, a^\dagger a}_{\text{mode}}
     + \underbrace{g(\sigma_+ a + \sigma_- a^\dagger)}_{\text{JC coupling}} $$

Physics note: composing the *dipole* coupling $g\,\sigma_x(a + a^\dagger)$ instead would give
the quantum **Rabi** model; Jaynes-Cummings is the post-RWA interaction, written directly as its
own term below. The framework composes literally -- choosing the frame/approximation is physics,
and all terms added together must live in the same frame (terms can carry a `frame=` tag; mixing
tags warns).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt

from htdse import (Operator, term, jump, hconj, HamiltonianEvolution,
                   LindbladEvolution, compare_over, fidelity, ket, otimes, quiet)
from htdse.core.plotting import plot_populations
from htdse.submodules.spin import sigma_z, sigma_x, sigma_plus
from htdse.submodules.harmonic_oscillator import annihilation, number_operator, fock

n_max = 12
a, n_op = annihilation(n_max), number_operator(n_max)
w0 = w = 1.3   # resonant
g = 0.11

atom = term(0.5 * w0 * sigma_z, on="spin", name="atom")
mode = term(w * n_op, on="mode", name="mode")
jc   = hconj(term({"spin": sigma_plus, "mode": a}, coeff=g, name="jc"))  # g(s+ a) + h.c.

H = atom + mode + jc
print(H)   # named groups + registry -- no matrix yet
print("materialized dimension:", H.hamiltonian(0.0).shape)

## Vacuum Rabi oscillation

Start in $|e, 0\rangle$. On resonance the composed model must give
$P_{e,0}(t) = \cos^2(gt)$ -- a known analytic result, so this doubles as a correctness check
of the composition. The term-layer Hamiltonian carries its own `subsystems` registry, so the
evolution (and `trace_out`) need no separate dims dict.

In [ ]:
psi0 = Operator(otimes(ket("0"), fock(0, n_max)))   # |e, 0>
ev = HamiltonianEvolution(H, psi0)                   # subsystems come from H itself

ts = np.linspace(0, 2 * np.pi / g, 200)
with quiet():
    psis = ev.state_at(ts)
P_e0 = np.abs(psis @ otimes(ket("0"), fock(0, n_max)).conj()) ** 2

plt.plot(ts, P_e0, label=r"$P_{e,0}(t)$ (composed model)")
plt.plot(ts, np.cos(g * ts) ** 2, "k--", lw=1, label=r"$\cos^2(gt)$")
plt.xlabel("t"); plt.ylabel("population"); plt.legend(); plt.show()

print("max deviation from cos^2(gt):", np.max(np.abs(P_e0 - np.cos(g * ts) ** 2)))

# reduced spin state, batched over all 200 times in one call
with quiet():
    rho_spin = ev.trace_out("mode", t=ts)
print("reduced-state trajectory:", rho_spin.shape)

## The swap-out workflow: replace the drive, keep the model

Named term groups are handles. Build the *target* model once; the *realized* model is the same
object with one entry swapped -- an amplitude-erred, detuned drive here. This is the
target-vs-reality comparison without writing a second Hamiltonian from scratch.

In [ ]:
Omega = 0.2
drive       = term(0.5 * Omega * sigma_x, on="spin", name="drive")
drive_noisy = term(0.5 * Omega * 1.05 * sigma_x, on="spin", name="drive") \
            + term(0.02 * sigma_z, on="spin", name="detuning")

model    = atom + mode + jc + drive
realized = model.replace(drive=drive_noisy)

with quiet():
    F = compare_over(ts,
                     HamiltonianEvolution(model, psi0),
                     HamiltonianEvolution(realized, psi0),
                     metric=fidelity)

plt.plot(ts, F)
plt.xlabel("t"); plt.ylabel("state fidelity target vs realized")
plt.title("Cost of a 5% amplitude error + detuning on the drive"); plt.show()

## Dissipation composes the same way

A Lindblad channel is just another named group (`htdse.jump`), embedded on the joint space by
name like any coherent term -- so "swap the ideal drive for a noisy *and lossy* one" is still
one `replace()`:

```python
lossy = drive_noisy + jump(a, on="mode", coeff=np.sqrt(gamma), name="mode_decay")
open_realized = model.replace(drive=lossy)
rho = LindbladEvolution(open_realized, rho0).state_at(ts)   # jump ops ride along
```

`LindbladEvolution` picks the jumps up automatically; handing a dissipative model to a
closed-system evolution class raises instead of silently ignoring the dissipation.